## Prepare data
- Load data (run/runs) from pickle (.pkl) file
- Normalize run/runs

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

from analysis import load_run, normalize_run

FOLDER = "26-02-26--11_04_41/mnist-accuracy_only-3-1.0-3-1.0-False-False.pkl"

data_dir = REPO_ROOT / "experiment" / "data" / "experimentData"
if FOLDER:
    data_dir = data_dir / FOLDER

run = load_run(data_dir)

# Normalize units: wei → ETH, ratio → %
run = normalize_run(run)


## Access data

In [2]:
# metadata = run.metadata
rounds_global = run.rounds_global
users = run.rounds_users
votes = run.votes
receipts = run.receipts

## Display data

In [3]:
votes

,experiment_id,round,giver_id,receiver_id,vote_score,giver_address,receiver_address
0,mnist-accuracy_only-3-1.0-3-1.0-False-False,1,0,1,0,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,0xc45595219926EC8Ba6892f1AE526d76cdAb7e1E2
1,mnist-accuracy_only-3-1.0-3-1.0-False-False,1,0,2,1,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,0x9f886a6bAD6DfD9409e05eE6743184feE0338d01
2,mnist-accuracy_only-3-1.0-3-1.0-False-False,1,0,3,0,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,0x1996Be88f24c47FA5C10D70FaA3489c9F9EcA0cd
3,mnist-accuracy_only-3-1.0-3-1.0-False-False,1,0,4,0,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,0x1930426B96FD9D0Ca399436995100315E3a37118
4,mnist-accuracy_only-3-1.0-3-1.0-False-False,1,0,5,1,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,0xcA8c42e4A7725bbd27d2d6911c579cc8f6d51913
...,...,...,...,...,...,...,...
135,mnist-accuracy_only-3-1.0-3-1.0-False-False,5,3,5,-1,0x1996Be88f24c47FA5C10D70FaA3489c9F9EcA0cd,0xcA8c42e4A7725bbd27d2d6911c579cc8f6d51913
136,mnist-accuracy_only-3-1.0-3-1.0-False-False,5,5,0,0,0xcA8c42e4A7725bbd27d2d6911c579cc8f6d51913,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276
137,mnist-accuracy_only-3-1.0-3-1.0-False-False,5,5,1,0,0xcA8c42e4A7725bbd27d2d6911c579cc8f6d51913,0xc45595219926EC8Ba6892f1AE526d76cdAb7e1E2
138,mnist-accuracy_only-3-1.0-3-1.0-False-False,5,5,2,0,0xcA8c42e4A7725bbd27d2d6911c579cc8f6d51913,0x9f886a6bAD6DfD9409e05eE6743184feE0338d01


## Options with data

## Merge users with their votes

In [4]:
# We merge users with votes
# Left: votes
# Right: users

votes_with_receiver = votes.merge(
    users[["experiment_id", "round", "user_id", "behavior", "role"]], # Rows we want from user
    left_on=["experiment_id", "round", "receiver_id"], # Rows on left to match
    right_on=["experiment_id", "round", "user_id"], # Rows on right to match
    how="left" # Join type
).rename(columns={
    "behavior": "receiver_behavior",
    "role": "receiver_role"
}).drop(columns=["user_id"]) # Redundant

# Six partitipants, that gives 5 votes over 5 rounds = 120 entries, but I see 150. But some participants might be kicked.

votes_with_receiver


,experiment_id,round,giver_id,receiver_id,vote_score,giver_address,receiver_address,receiver_behavior,receiver_role
0,mnist-accuracy_only-3-1.0-3-1.0-False-False,1,0,1,0,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,0xc45595219926EC8Ba6892f1AE526d76cdAb7e1E2,good,good
1,mnist-accuracy_only-3-1.0-3-1.0-False-False,1,0,2,1,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,0x9f886a6bAD6DfD9409e05eE6743184feE0338d01,good,good
2,mnist-accuracy_only-3-1.0-3-1.0-False-False,1,0,3,0,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,0x1996Be88f24c47FA5C10D70FaA3489c9F9EcA0cd,good,good
3,mnist-accuracy_only-3-1.0-3-1.0-False-False,1,0,4,0,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,0x1930426B96FD9D0Ca399436995100315E3a37118,good,bad
4,mnist-accuracy_only-3-1.0-3-1.0-False-False,1,0,5,1,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,0xcA8c42e4A7725bbd27d2d6911c579cc8f6d51913,good,freerider
...,...,...,...,...,...,...,...,...,...
135,mnist-accuracy_only-3-1.0-3-1.0-False-False,5,3,5,-1,0x1996Be88f24c47FA5C10D70FaA3489c9F9EcA0cd,0xcA8c42e4A7725bbd27d2d6911c579cc8f6d51913,freerider,freerider
136,mnist-accuracy_only-3-1.0-3-1.0-False-False,5,5,0,0,0xcA8c42e4A7725bbd27d2d6911c579cc8f6d51913,0x909362B1290E04f28ab421261B8Bdd1E5ffBA276,good,good
137,mnist-accuracy_only-3-1.0-3-1.0-False-False,5,5,1,0,0xcA8c42e4A7725bbd27d2d6911c579cc8f6d51913,0xc45595219926EC8Ba6892f1AE526d76cdAb7e1E2,good,good
138,mnist-accuracy_only-3-1.0-3-1.0-False-False,5,5,2,0,0xcA8c42e4A7725bbd27d2d6911c579cc8f6d51913,0x9f886a6bAD6DfD9409e05eE6743184feE0338d01,good,good


## Jupyter notes

Use, the following to display all columns

```python
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
```
